# Rețele neuronale și deep learning — Cursul 12: Modele lingvistice mari în finanțe și formularea instrucțiunilor
Prof. dr. Daniel Traian Pele — Academia de Studii Economice din București (ASE); Academia Română, Institutul de Prognoză Economică (IPE)

[Deschide în Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L12_notebook_ro.ipynb)

Acest notebook reproduce cifrele și graficele cursului pe datele reale din `../data/clean` (vezi `../data/SOURCES.md`). Numerele secțiunilor (§) corespund indicațiilor *Exercițiu practic* de pe slide-uri.

In [1]:
# Google Colab: copiem repo-ul cursului, ca datele și rezultatele salvate să fie găsite
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [2]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

device: mps


## §1 Tokeni și perplexitate

In [3]:
RES = '../lectures/L12/results/'
import json, re, torch
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForCausalLM
PB = pd.read_csv(DATA + 'financial_phrasebank.csv')
SP = pd.read_csv(DATA + 'ecb_speeches_2022plus.csv', parse_dates=['date'])
LAB = ['positive', 'negative', 'neutral']
print(len(PB), 'sentences;', len(SP), 'ECB speeches')

4838 sentences; 400 ECB speeches


In [4]:
name = 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(name)
lm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16 if DEVICE != 'cpu' else torch.float32).to(DEVICE).eval()
words = PB.sentence.str.split().str.len().sum()
toks = sum(len(tok.encode(s)) for s in PB.sentence)
print('Qwen tokens per word:', round(toks / words, 2))
print(tok.tokenize(PB.sentence[0]))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen tokens per word: 1.35
['(', 'ĠA', 'Ġspokesperson', 'Ġtold', 'ĠW', 'WD', 'Ġto', 'Ġexpect', 'Ġa', 'Ġ', '5', '0', '-', '5', '0', 'Ġmix', 'Ġof', 'Ġclothing', 'Ġand', 'Ġhome', 'Ġdecor', 'Ġ.', 'Ġ)']


In [5]:
txt = SP.sort_values('date').contents.iloc[-1][:4000]
ids = tok(txt, return_tensors='pt').input_ids[:, :512].to(DEVICE)
with torch.no_grad():
    nll = torch.nn.functional.cross_entropy(lm(ids).logits[0, :-1].float(), ids[0, 1:])
print('perplexity of Qwen2.5-0.5B on the latest ECB speech (512 tokens):', round(float(torch.exp(nll)), 2))
J = json.load(open(RES + 'lm_outputs.json'))
print({s: round(float(np.exp(v['new'])), 2) for s, v in J['ppl_nll'].items()})

perplexity of Qwen2.5-0.5B on the latest ECB speech (512 tokens): 14.16
{'0.5B': 14.71, '1.5B': 10.63, '3B': 9.43}


## §2 Probabilitățile următorului token și temperatura

In [6]:
prompt = 'Inflation in the euro area is expected to'
ids = tok(prompt, return_tensors='pt').input_ids.to(DEVICE)
with torch.no_grad():
    z = lm(ids).logits[0, -1].float().cpu()
for Tm in (0.5, 1.0, 2.0):
    p = torch.softmax(z / Tm, -1)
    top = torch.topk(p, 5)
    print(Tm, [(tok.decode([i]), round(float(v), 3)) for i, v in zip(top.indices, top.values)])

0.5 [(' be', 0.528), (' rise', 0.151), (' remain', 0.142), (' slow', 0.046), (' reach', 0.039)]
1.0 [(' be', 0.191), (' rise', 0.102), (' remain', 0.099), (' slow', 0.056), (' reach', 0.052)]
2.0 [(' be', 0.02), (' rise', 0.015), (' remain', 0.015), (' slow', 0.011), (' reach', 0.011)]


§ *Exercițiu practic*: schimbați promptul cu o propoziție proprie și urmăriți cum se modifică entropia cu temperatura.

## §3 Un prompt ca un clasificator (live pe 60 de propoziții, rezultatele complete din analiză)

In [7]:
ids_lab = {l: sorted({tok.encode(v)[0] for v in (l, ' ' + l, l.capitalize(), ' ' + l.capitalize())}) for l in LAB}
INSTR = ('Classify the sentiment of this sentence from financial news, from the point of view of an investor in the company. '
         'Answer with one word: positive, negative or neutral.')
def zero_shot(sentences):
    out = []
    for s in sentences:
        msgs = [{'role': 'system', 'content': 'You are a careful financial analyst.'}, {'role': 'user', 'content': INSTR + '\n\nSentence: ' + s + '\nAnswer:'}]
        ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors='pt').input_ids.to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(lm(ids).logits[0, -1].float(), -1)
        out.append(LAB[int(np.argmax([float(p[ids_lab[l]].sum()) for l in LAB]))])
    return out
P = pd.read_csv(RES + 'predictions_pb.csv')
small = P.sample(60, random_state=SEED)
pred = zero_shot(small.text.tolist())
print('live zero-shot 0.5B, 60 sentences, accuracy:', round(accuracy_score(small.label, pred), 3))

live zero-shot 0.5B, 60 sentences, accuracy: 0.683


In [8]:
def f1(df, m):
    return f1_score(df.label, [LAB[i] for i in df[[f'{m}_{l}' for l in LAB]].values.argmax(1)], labels=LAB, average='macro')
for m in ['tfidf', 'zs_0.5B', 'zs_1.5B', 'zs_3B', 'fs_1.5B', 'fs_3B', 'cot_3B', 'probe']:
    print(m, round(100 * f1(P, m), 1))

tfidf 86.9
zs_0.5B 31.0
zs_1.5B 85.1
zs_3B 92.2
fs_1.5B 85.7
fs_3B 91.8
cot_3B 88.4
probe 87.5


## §4 Exemple few-shot și sensibilitatea la prompt

In [9]:
L = pd.read_csv(RES + 'fewshot_labels_pb.csv')
for s in ('1.5B', '3B'):
    print(s, [round(100 * np.mean([f1_score(L.label, L[f'{s}_k{k}_s{sd}'], labels=LAB, average='macro') for sd in (0, 1, 2)]), 1) for k in (0, 1, 2, 4, 8)])
S = pd.read_csv(RES + 'sensitivity_labels_pb.csv')
print('3B, five wordings:', [round(100 * f1_score(S.label, S[f'3B_i{i}'], labels=LAB, average='macro'), 1) for i in range(1, 6)])

1.5B [np.float64(85.1), np.float64(87.3), np.float64(86.0), np.float64(85.9), np.float64(84.1)]
3B [np.float64(92.2), np.float64(89.5), np.float64(90.4), np.float64(92.1), np.float64(91.3)]
3B, five wordings: [92.2, 91.4, 88.5, 90.0, 90.7]


## §5 Chain-of-thought

In [10]:
msgs = [{'role': 'user', 'content': INSTR.split(' Answer')[0] + ' First explain your reasoning in at most two short sentences, then give the final answer as Label: <positive|negative|neutral>.\n\nSentence: ' + small.text.iloc[0]}]
ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors='pt').input_ids.to(DEVICE)
with torch.no_grad():
    g = lm.generate(ids, max_new_tokens=80, do_sample=False)
print(small.text.iloc[0])
print(tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True))

Operating profit was EUR 139.7 mn , up 23 % from EUR 113.8 mn .
The sentiment of this sentence is positive.

Reasoning:
- The sentence mentions operating profit, which is typically seen as a positive metric for a company.
- It also indicates a percentage increase, which is generally viewed positively in financial terms.
- There are no negative or neutral indicators present in the text.

Final Answer: Positive


## §6 Calibrare, acordul adnotatorilor, cost

In [11]:
def ece(m, bins=10):
    Pm = P[[f'{m}_{l}' for l in LAB]].values
    conf = Pm.max(1)
    ok = np.array([LAB[i] == t for i, t in zip(Pm.argmax(1), P.label)])
    edges = np.linspace(1 / 3, 1, bins + 1)
    e = 0.0
    for a, b in zip(edges[:-1], edges[1:]):
        msk = (conf >= a) & (conf <= b if b == 1 else conf < b)
        e += msk.mean() * abs(conf[msk].mean() - ok[msk].mean()) if msk.sum() else 0
    return round(100 * e, 1)
print({m: ece(m) for m in ('zs_3B', 'fs_3B', 'tfidf')})
A = pd.read_csv(RES + 'agreement_labels.csv')
print(A.groupby('agreement').apply(lambda g: round(100 * accuracy_score(g.label, g.zs_3B), 1)))
print(J['ms_per_item']['pb'])

{'zs_3B': np.float64(4.7), 'fs_3B': np.float64(3.6), 'tfidf': np.float64(5.4)}
agreement
50    53.7
66    65.7
75    77.2
dtype: float64
{'majority': 0.0, 'tfidf': 2.5, 'zs_0.5B': 19.54, 'zs_1.5B': 28.51, 'zs_3B': 48.5, 'fs_1.5B': 22.04, 'fs_3B': 40.78, 'cot_3B': 389.48, 'probe': 25.37}


## §7 Regăsirea în discursurile BCE: TF-IDF și BM25, live

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
def body(r):
    c = r.contents
    h = f'SPEECH {r.title} {r.subtitle}'
    c = c[len(h):] if c.startswith(h) else c
    m = re.search(r'\b(19|20)\d{2}\b', c[:120])
    return c[m.end():].strip() if m else c
SP = SP.sort_values('date').reset_index(drop=True)
chunks, owner = [], []
for i, r in SP.iterrows():
    w = body(r).split()
    for j in range(0, max(1, len(w) - 50), 150):
        chunks.append(' '.join(w[j:j + 200])); owner.append(i)
        if j + 200 >= len(w):
            break
owner = np.array(owner)
vec = TfidfVectorizer(stop_words='english', sublinear_tf=True)
C = vec.fit_transform(chunks)
S = (vec.transform(SP.title) @ C.T).toarray()
rank = [int(np.where(pd.Series(S[q]).groupby(owner).max().sort_values(ascending=False).index.values == q)[0][0]) + 1 for q in range(len(SP))]
print(len(chunks), 'chunks; TF-IDF recall@1 =', round(100 * np.mean(np.array(rank) <= 1), 1), '%')
R = pd.read_csv(RES + 'rag_ranks.csv')
print({m: round(100 * float((R[f'rank_{m}'] <= 1).mean()), 1) for m in ('tfidf', 'bm25', 'dense', 'hybrid')})

8400 chunks; TF-IDF recall@1 = 35.2 %
{'tfidf': 35.2, 'bm25': 37.0, 'dense': 44.0, 'hybrid': 50.7}


## §8 RAG față de răspunsurile fără documente

In [13]:
Q = pd.read_csv(RES + 'rag_qa.csv')
print(pd.DataFrame({m: Q[m].value_counts(normalize=True).mul(100).round(1) for m in ('closed', 'rag', 'oracle')}))

         closed   rag  oracle
abstain     NaN  14.4     1.2
correct    13.8  81.2    98.8
wrong      86.2   4.4     NaN


## §9 Testarea retrospectivă a LLM-VaR

In [14]:
from scipy import stats
V = pd.read_csv(RES + 'llmvar_sp500.csv', parse_dates=['date']).set_index('date')
def kupiec(h, a):
    n, k = len(h), int(h.sum()); p = k / n
    lr = -2 * ((n - k) * np.log(1 - a) + k * np.log(a) - (n - k) * np.log(1 - p) - k * np.log(p)) if k else -2 * n * np.log(1 - a)
    return round(float(1 - stats.chi2.cdf(lr, 1)), 3)
for m in ('llm', 'hs', 'garch'):
    for a, s in ((0.05, '05'), (0.01, '01')):
        h = (V.r < V[f'{m}_{s}']).values
        print(m, s, 'violations', int(h.sum()), 'expected', round(a * len(h), 1), 'Kupiec p', kupiec(h, a))

llm 05 violations 28 expected 23.0 Kupiec p 0.295
llm 01 violations 4 expected 4.6 Kupiec p 0.777
hs 05 violations 23 expected 23.0 Kupiec p 0.991
hs 01 violations 6 expected 4.6 Kupiec p 0.528
garch 05 violations 35 expected 23.0 Kupiec p 0.016
garch 01 violations 9 expected 4.6 Kupiec p 0.067
